# W2 보조 — 작은 학습 데모

본 자료의 본 노트북에 들어가기 전, 핵심 흐름을 더 작은 예시로 확인합니다.

## 0. 환경 준비

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('../../COMMON/shared/week2/helpers').resolve()))

import numpy as np
import matplotlib.pyplot as plt
import torch
from dr_utils import load_volume, porosity, predict_linear_k, eval_targets, ssim_3d_mean, setup_plot_style, ORANGE
from model_utils import UNetMini, count_parameters, SliceDataset, train_quick, evaluate_model, TRAINING_PRESETS
setup_plot_style()

DATA = Path('../../COMMON/shared/week2/data').resolve()
bb = load_volume(DATA / 'BB_256.bin')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'BB: shape={bb.shape}, φ={porosity(bb)*100:.2f}%, device={DEVICE}')

## 예시 1 — UNet 의 한 번 forward pass

모델을 만들어 입력을 한 번만 통과시켜봅니다. 학습은 안 함.

In [ ]:
model = UNetMini(in_ch=2, base=8)
print(f'params: {count_parameters(model):,}')

# Random 입력 (B=1, 2채널, 64x64)
x = torch.randn(1, 2, 64, 64)
with torch.no_grad():
    y = model(x)
print(f'input  shape: {x.shape}')
print(f'output shape: {y.shape}  range: [{y.min():.3f}, {y.max():.3f}]')

**관찰**: 출력이 (1, 1, 64, 64) — 입력 한 배치당 한 슬라이스 예측. 출력 값이 0~1 범위인 이유는 마지막에 sigmoid가 있기 때문.

**시도**: `base=16` 또는 `base=32` 로 바꾸고 파라미터 수 변화 확인.

## 예시 2 — Dataset 의 한 sample 확인

In [ ]:
ds = SliceDataset(bb, k=1, patch_size=64, n_patches_per_triplet=2)
print(f'Dataset size: {len(ds)} samples')

x, y = ds[0]
print(f'input  x.shape = {x.shape}')
print(f'target y.shape = {y.shape}')

fig, axes = plt.subplots(1, 3, figsize=(10, 4))
axes[0].imshow(x[0], cmap='gray'); axes[0].set_title('입력 ch0 (이웃 t−k)'); axes[0].axis('off')
axes[1].imshow(x[1], cmap='gray'); axes[1].set_title('입력 ch1 (이웃 t+k)'); axes[1].axis('off')
axes[2].imshow(y[0], cmap='gray'); axes[2].set_title('정답 (가운데 슬라이스 t)'); axes[2].axis('off')
plt.tight_layout(); plt.show()

## 예시 3 — 미니 학습 (1~2분)

정식 학습은 본 자료 fast preset (10분) 이지만, 여기서는 더 짧게 5 epoch만 돌려서 흐름을 봅니다.

In [ ]:
from torch.utils.data import DataLoader
import torch.nn as nn

model = UNetMini(in_ch=2, base=8).to(DEVICE)
ds = SliceDataset(bb, k=1, patch_size=64, n_patches_per_triplet=1)
loader = DataLoader(ds, batch_size=4, shuffle=True, num_workers=0)
criterion = nn.L1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

history = []
for ep in range(5):
    model.train()
    total, n = 0.0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        total += loss.item() * x.size(0); n += x.size(0)
    history.append(total / n)
    print(f'  epoch {ep+1}/5  loss={total/n:.4f}')

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history, marker='o', color=ORANGE, lw=2)
plt.xlabel('Epoch'); plt.ylabel('L1 loss')
plt.title('mini 학습 곡선 (5 epoch)')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

**관찰**: 5 epoch 만에도 loss 가 줄어드는 게 보입니다. 실제 학습은 20~100 epoch 사용.

**시도**: `lr=1e-3` 을 `lr=1e-1` (너무 큼) 로 바꿔보고 loss 가 발산(NaN) 하는 것을 관찰.

## 다음 단계

흐름이 익숙해졌으면 본 자료의 `W2_deep_learning_intro.ipynb` 를 fast preset 으로 정식 학습해보세요.